# Muhammad CNN

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

woy
woy2
Resuming download from 115343360 bytes (247808853 bytes left)...
Resuming download to /home/nayaka/.cache/kagglehub/datasets/puneet6060/intel-image-classification/2.archive (115343360/363152213) bytes left.


100%|██████████| 346M/346M [01:32<00:00, 2.67MB/s]

Extracting files...


Path to dataset files: /home/nayaka/.cache/kagglehub/datasets/puneet6060/intel-image-classification/versions/2


In [9]:
import os
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

ModuleNotFoundError: No module named 'tensorflow'

In [7]:
DATASET_ROOT = Path("~/.cache/kagglehub/datasets/puneet6060/intel-image-classification/versions/2")

TRAIN_DIR = DATASET_ROOT / "seg_train" / "seg_train"
VAL_DIR = DATASET_ROOT / "seg_test" / "seg_test"
TEST_DIR = DATASET_ROOT / "seg_test" / "seg_test"

# bwt hiperparameterrrz
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 20

ARTIFACTS_DIR = Path("../artifacts/cnn")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"root dataset: {DATASET_ROOT}")
print(f"cek training path: {TRAIN_DIR.exists()}")

NameError: name 'Path' is not defined

In [ ]:
# === Data loading ===
def build_dataset(data_dir, shuffle=True):
    return tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels="inferred",
        label_mode="int",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )

train_ds = build_dataset(TRAIN_DIR, shuffle=True)
val_ds = build_dataset(VAL_DIR, shuffle=False)
test_ds = build_dataset(TEST_DIR, shuffle=False)

# Performance optimizations
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
class_names

In [ ]:
# === Model builder ===
def build_cnn(
    input_shape=(150, 150, 3),
    num_classes=6,
    filters=(32, 64, 128),
    dense_units=128,
    use_global_pooling=False,
):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 255)(inputs)

    for f in filters:
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)

    if use_global_pooling:
        x = layers.GlobalAveragePooling2D()(x)
    else:
        x = layers.Flatten()(x)

    x = layers.Dense(dense_units, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    return model

In [ ]:
# === Training (Bagian 3) ===
import time
import json

model = build_cnn(
    input_shape=IMG_SIZE + (3,),
    num_classes=NUM_CLASSES,
    filters=(32, 64, 128),
    dense_units=128,
    use_global_pooling=False,
 )

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

t0 = time.perf_counter()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
 )
train_time_sec = time.perf_counter() - t0

# Save weights
weights_path = ARTIFACTS_DIR / "cnn_weights.h5"
model.save_weights(weights_path)

# Plot training history
def plot_history(hist):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist.history["loss"], label="train")
    axes[0].plot(hist.history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(hist.history["accuracy"], label="train")
    axes[1].plot(hist.history["val_accuracy"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].legend()

    plt.show()

plot_history(history)

# === Evaluation (Bagian 4) ===
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# Classification report and confusion matrix
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_pred_probs = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# === Experiments (Bagian 3/4) ===
# Sweep several architecture variations and record accuracy + time.
experiment_configs = [
    {"filters": (32, 64), "dense_units": 64, "use_global_pooling": False},
    {"filters": (32, 64), "dense_units": 128, "use_global_pooling": False},
    {"filters": (32, 64, 128), "dense_units": 128, "use_global_pooling": False},
    {"filters": (32, 64, 128), "dense_units": 256, "use_global_pooling": False},
    {"filters": (32, 64, 128), "dense_units": 256, "use_global_pooling": True},
    {"filters": (32, 64, 128, 256), "dense_units": 256, "use_global_pooling": True},
]

experiment_results = []
experiments_dir = ARTIFACTS_DIR / "experiments"
experiments_dir.mkdir(parents=True, exist_ok=True)

for idx, cfg in enumerate(experiment_configs, start=1):
    exp_model = build_cnn(
        input_shape=IMG_SIZE + (3,),
        num_classes=NUM_CLASSES,
        filters=cfg["filters"],
        dense_units=cfg["dense_units"],
        use_global_pooling=cfg["use_global_pooling"],
    )
    exp_model.compile(
        optimizer=keras.optimizers.Adam(),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    t0 = time.perf_counter()
    exp_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=0,
    )
    train_time_sec = time.perf_counter() - t0

    val_loss, val_acc = exp_model.evaluate(val_ds, verbose=0)
    cfg_tag = f"exp_{idx}_f{'-'.join(map(str, cfg['filters']))}_d{cfg['dense_units']}_gp{int(cfg['use_global_pooling'])}"
    weights_path = experiments_dir / f"{cfg_tag}_weights.h5"
    exp_model.save_weights(weights_path)

    experiment_results.append({
        **cfg,
        "val_loss": float(val_loss),
        "val_acc": float(val_acc),
        "train_time_sec": float(train_time_sec),
        "weights_path": str(weights_path),
    })

results_path = experiments_dir / "experiment_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(experiment_results, f, indent=2)

experiment_results